In [1]:
import requests

# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
test_urls = {
    'Powell-Thyne CY': 'http://www.uky.edu/~clthyn2/coup_data/powell_thyne_ccode_year.txt',
    'UNODC Homicide OWID': 'https://ourworldindata.org/grapher/homicide-rate-unodc.csv?v=1&csvType=full&useColumnShortNames=false',
    'IRENA OWID': 'https://ourworldindata.org/grapher/share-electricity-renewables.csv?v=1&csvType=full&useColumnShortNames=false',
    'Basel AML downloads': 'https://index.baselgovernance.org/downloads',
}

for name, url in test_urls.items():
    try:
        response = requests.head(url, timeout=15, allow_redirects=True)
        content_type = response.headers.get('Content-Type', '')[:40]
        print(f"{name}: {response.status_code} [{content_type}]")
    except Exception as e:
        print(f"{name}: ERROR — {e}")

Powell-Thyne CY: ERROR — HTTPSConnectionPool(host='www.uky.edu', port=443): Read timed out. (read timeout=15)
UNODC Homicide OWID: 200 [text/csv]
IRENA OWID: 200 [text/csv]
Basel AML downloads: 200 [text/html;charset=utf-8]


In [2]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING
import requests

# Test Powell-Thyne with longer timeout
try:
    response = requests.get(
        'http://www.uky.edu/~clthyn2/coup_data/powell_thyne_ccode_year.txt',
        timeout=60
    )
    print(f"Powell-Thyne: {response.status_code}, size={len(response.content)/1024:.1f}KB")
    print(response.text[:200])
except Exception as e:
    print(f"Powell-Thyne: ERROR — {e}")

# Scrape Basel AML downloads page for CSV link
response = requests.get('https://index.baselgovernance.org/downloads', timeout=30)
import re
csv_links = re.findall(r'https?://[^\s"\'<>]+\.(?:csv|xlsx)', response.text)
print(f"\nBasel AML file links found: {csv_links[:5]}")

Powell-Thyne: 200, size=691.7KB
ccode	abbrev	country	year	ccode_gw	ccode_polity	coup1	coup2	coup3	coup4	date1	date2	date3	date4	version
2	"USA"	"United States of America"	1950			0	0	0	0					"V2026.01.13"
2	"USA"	"United States of 

Basel AML file links found: []


In [3]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Check IRENA OWID data structure
import io
import pandas as pd

irena_response = requests.get(
    'https://ourworldindata.org/grapher/share-electricity-renewables.csv?v=1&csvType=full&useColumnShortNames=false',
    timeout=30
)
irena_df = pd.read_csv(io.StringIO(irena_response.text))
print(f"IRENA OWID shape: {irena_df.shape}")
print(f"Columns: {list(irena_df.columns)}")
print(f"Years: {irena_df['Year'].min()} — {irena_df['Year'].max()}")
print(irena_df.head(3))

# Test Basel AML direct CSV URL patterns
basel_urls = [
    "https://index.baselgovernance.org/api/assets/basel-aml-index-2025.csv",
    "https://index.baselgovernance.org/scores.csv",
    "https://index.baselgovernance.org/api/scores.csv",
]
for url in basel_urls:
    r = requests.head(url, timeout=10, allow_redirects=True)
    print(f"\nBasel {url.split('/')[-1]}: {r.status_code} [{r.headers.get('Content-Type','')[:40]}]")

IRENA OWID shape: (7872, 4)
Columns: ['Entity', 'Code', 'Year', 'Renewables']
Years: 1985 — 2025
          Entity Code  Year  Renewables
0  ASEAN (Ember)  NaN  2000   19.334143
1  ASEAN (Ember)  NaN  2001   19.055025
2  ASEAN (Ember)  NaN  2002   17.666613

Basel basel-aml-index-2025.csv: 403 [application/json; charset=utf-8]

Basel scores.csv: 404 [text/html;charset=utf-8]

Basel scores.csv: 404 [application/json; charset=utf-8]


In [4]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Find correct IRENA capacity URL on OWID
irena_capacity_urls = [
    "https://ourworldindata.org/grapher/installed-renewable-energy-capacity-gigawatts.csv?v=1&csvType=full&useColumnShortNames=false",
    "https://ourworldindata.org/grapher/renewable-energy-capacity-by-technology.csv?v=1&csvType=full&useColumnShortNames=false",
    "https://ourworldindata.org/grapher/irena-renewable-energy-capacity.csv?v=1&csvType=full&useColumnShortNames=false",
]

for url in irena_capacity_urls:
    r = requests.head(url, timeout=10, allow_redirects=True)
    print(f"{url.split('/')[-1].split('.')[0]}: {r.status_code} [{r.headers.get('Content-Type','')[:30]}]")

# Fetch Basel AML downloads page fully
basel_page = requests.get('https://index.baselgovernance.org/downloads', timeout=30)
# Look for any download links
all_links = re.findall(r'href=["\']([^"\']+)["\']', basel_page.text)
data_links = [l for l in all_links if any(ext in l.lower() for ext in ['.csv', '.xlsx', '.xls', 'download', 'data'])]
print(f"\nBasel data links: {data_links[:10]}")

installed-renewable-energy-capacity-gigawatts: 404 [application/json; charset=utf-]
renewable-energy-capacity-by-technology: 404 [application/json; charset=utf-]
irena-renewable-energy-capacity: 404 [application/json; charset=utf-]

Basel data links: ['https://aml-dev.baselgovernance.org/downloads', '/downloads', '/downloads', '/downloads']


In [5]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Test correct IRENA capacity URL
irena_url = "https://ourworldindata.org/grapher/installed-global-renewable-energy-capacity-by-technology.csv?v=1&csvType=full&useColumnShortNames=false"
r = requests.get(irena_url, timeout=30)
print(f"IRENA capacity: {r.status_code}, size={len(r.content)/1024:.1f}KB")
if r.status_code == 200:
    df_irena = pd.read_csv(io.StringIO(r.text))
    print(f"Shape: {df_irena.shape}, columns: {list(df_irena.columns)}")
    print(f"Years: {df_irena['Year'].min()} — {df_irena['Year'].max()}")

# Check Basel AML downloads page for JSON data containing file URLs
basel_page = requests.get('https://index.baselgovernance.org/downloads', timeout=30)
json_links = re.findall(r'"url"\s*:\s*"([^"]+\.(?:csv|xlsx))"', basel_page.text)
asset_links = re.findall(r'/api/assets/[a-f0-9\-]+', basel_page.text)
print(f"\nBasel JSON CSV links: {json_links[:5]}")
print(f"Basel asset links: {asset_links[:5]}")

IRENA capacity: 200, size=14.4KB
Shape: (475, 3), columns: ['Entity', 'Year', 'Installed capacity for different renewable technologies']
Years: 2000 — 2024

Basel JSON CSV links: []
Basel asset links: ['/api/assets/88bff09e-0505-48dc-8819-f976a9ebf78e', '/api/assets/1cdb5e5f-f4c2-4738-918b-f2c4271c6313', '/api/assets/9cb0e18f-2fdf-4429-a268-b321479d9a30', '/api/assets/4268694f-14a3-4ff0-b03b-e678f4b04a73', '/api/assets/8ebee704-b518-42ca-8de5-c67dbc71a18a']


In [6]:
# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING

# Test Basel AML asset links
for asset in asset_links[:3]:
    url = f"https://index.baselgovernance.org{asset}"
    r = requests.head(url, timeout=10, allow_redirects=True)
    print(f"{asset[-8:]}: {r.status_code} [{r.headers.get('Content-Type','')[:40]}]")

# Check if IRENA per-country renewable share works
irena_share_df = pd.read_csv(
    io.StringIO(requests.get(
        'https://ourworldindata.org/grapher/share-electricity-renewables.csv?v=1&csvType=full&useColumnShortNames=false',
        timeout=30
    ).text)
)
print(f"\nIRENA share: {irena_share_df.shape}")
print(f"Sample countries: {irena_share_df['Code'].dropna().unique()[:5]}")
print(f"Years: {irena_share_df['Year'].min()} — {irena_share_df['Year'].max()}")

a9ebf78e: 200 [image/jpeg]
271c6313: 200 [application/pdf]
479d9a30: 200 [application/pdf]

IRENA share: (7872, 4)
Sample countries: <StringArray>
['AFG', 'OWID_AFR', 'ALB', 'DZA', 'ASM']
Length: 5, dtype: str
Years: 1985 — 2025
